In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

#sns.set_style("whitegrid")
#sns.set_context("talk")
plt.rcParams['axes.facecolor'] = "#f5f5dc"

In [ ]:
df = pd.read_csv('train.csv')
df.set_index('id', inplace = True)
df

In [ ]:
# Zakładamy, że df to Twój DataFrame
# Wybieramy cechy numeryczne bez id i targetu
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = df.drop(columns=['FloodProbability']).columns

X = df[features]
y = df['FloodProbability']

# Skalowanie cech – przydatne dla Lasso i Ridge
scaler = StandardScaler()
X = scaler.fit_transform(X)

These 2 below are optional

In [ ]:
# top zmienne wg nachylenia
top_vars = [
    "PoliticalFactors",
    "DamsQuality",
    "RiverManagement",
    "Watersheds",
    "MonsoonIntensity",
    "PopulationScore",
    "Deforestation",
    "Encroachments",
    "Siltation",
    "AgriculturalPractices"
]

# funkcja do tworzenia cechy jako iloczynu zmiennych
def create_feature(df, vars_list, scale=False):
    df_temp = df[vars_list].copy()
    if scale:
        scaler = MinMaxScaler()
        df_temp[vars_list] = scaler.fit_transform(df_temp)
    feature = df_temp.prod(axis=1)
    return feature

# tworzymy 4 nowe cechy
df['feat_top3_scaled']  = create_feature(df, top_vars[:3], scale=True)
df['feat_top5_scaled']  = create_feature(df, top_vars[:5], scale=True)
df['feat_top10']        = create_feature(df, top_vars[:10], scale=True)

In [ ]:
# funkcja do tworzenia cechy jako sumy zmiennych
def create_feature_sum(df, vars_list, scale=False):
    df_temp = df[vars_list].copy()
    if scale:
        scaler = MinMaxScaler()
        df_temp[vars_list] = scaler.fit_transform(df_temp)
    # suma wszystkich kolumn
    feature = df_temp.sum(axis=1)
    return feature

# tworzymy 4 nowe cechy sumaryczne
df['feat_top3_sum_scaled']  = create_feature_sum(df, top_vars[:3], scale=False)
df['feat_top5_sum_scaled']  = create_feature_sum(df, top_vars[:5], scale=False)
df['feat_top10_sum']        = create_feature_sum(df, top_vars[:10], scale=False)

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, cross_val_predict
import numpy as np
import matplotlib.pyplot as plt

# Parameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0]
}

# Model na GPU
xgb = XGBRegressor(random_state=42, tree_method="hist", device="cuda", n_jobs=-1)

# Grid search
grid_search = GridSearchCV(
    xgb,
    param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    verbose=2,
    n_jobs=-1
)

grid_search.fit(X, y)
print("Best parameters (XGBoost):", grid_search.best_params_)

best_xgb = grid_search.best_estimator_

# Cross-validation predictions
y_pred = cross_val_predict(best_xgb, X, y, cv=5)

# Feature importance
importances = best_xgb.feature_importances_
feature_names = X.columns if hasattr(X, "columns") else [f"f{i}" for i in range(X.shape[1])]

plt.figure(figsize=(10,6))
plt.barh(feature_names, importances)
plt.xlabel("Feature Importance")
plt.title("XGBoost Feature Importance")
plt.show()

# Correlation plot
plt.figure(figsize=(6,6))
plt.scatter(y, y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--')
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title("XGBoost: Actual vs Predicted")
plt.show()